# 03 — Retrieval Strategies: Dense, Sparse e Hybrid

A estrategia de retrieval define como encontrar os chunks certos para responder uma pergunta.

## Tipos de Retrieval

| Tipo | Como funciona | Bom para | Ruim para |
|------|--------------|----------|----------|
| **Dense** | Similaridade de embeddings | Sinonimos, contexto | Keywords exatas |
| **Sparse (BM25)** | TF-IDF / word frequency | Keywords exatas | Sinonimos, contexto |
| **Hybrid** | Dense + Sparse combinados | Maioria dos casos | - |
| **MMR** | Max Marginal Relevance | Diversidade de resultados | Velocidade |

**Regra geral:** Hybrid retrieval supera dense-only em ~5-15% de recall na maioria dos benchmarks.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from rank_bm25 import BM25Okapi
import re

model = SentenceTransformer('all-MiniLM-L6-v2')
client = QdrantClient(host='localhost', port=6333)

# Corpus de documentos
corpus = [
    'Transformers usam mecanismo de self-attention para processar sequencias',
    'BERT e um modelo pre-treinado baseado em transformer para classificacao de texto',
    'GPT utiliza decoder-only transformer com autoregressive language modeling',
    'Embeddings de palavra capturam relacoes semanticas entre termos',
    'BM25 e um algoritmo de ranking baseado em frequencia de termos e TF-IDF',
    'Reciprocal Rank Fusion combina rankings de multiplos sistemas de busca',
    'O algoritmo HNSW cria um grafo hierarquico para busca aproximada eficiente',
    'Cosine similarity mede o angulo entre dois vetores no espaco',
    'RAG usa recuperacao de documentos para fundamentar respostas de LLMs',
    'Query expansion adiciona termos relacionados para melhorar o recall',
    'Cross-encoder re-ranking reordena resultados por relevancia mais precisa',
    'Chunk overlap preserva contexto entre segmentos adjacentes de texto',
    'Sentence transformers produzem embeddings semanticos de alta qualidade',
    'Qdrant armazena vetores com metadados e suporta filtragem eficiente',
    'MMR maximiza relevancia e minimiza redundancia nos resultados retornados',
]

# Indexar no Qdrant
embs = model.encode(corpus, normalize_embeddings=True)

client.recreate_collection('retrieval_demo', vectors_config=VectorParams(size=384, distance=Distance.COSINE))
points = [PointStruct(id=i, vector=embs[i].tolist(), payload={'texto': corpus[i]}) for i in range(len(corpus))]
client.upsert('retrieval_demo', points=points)

# BM25 index
corpus_tokenizado = [re.sub(r'[^a-z0-9 ]', '', doc.lower()).split() for doc in corpus]
bm25 = BM25Okapi(corpus_tokenizado)

print(f'Corpus: {len(corpus)} documentos indexados')

## 3.1 Dense Retrieval (Semantico)

In [ ]:
def dense_retrieve(query, top_k=5):
    q_vec = model.encode(query, normalize_embeddings=True)
    results = client.search('retrieval_demo', query_vector=q_vec.tolist(), limit=top_k, with_payload=True)
    return [(r.id, r.score, r.payload['texto']) for r in results]

# Casos onde dense se sai bem: busca semantica
queries_semanticas = [
    ('redes neurais para linguagem', 'Transformers, BERT, GPT — entende contexto'),
    ('encontrar itens similares rapidamente', 'HNSW, cosine similarity — captura intenção'),
]

print('DENSE RETRIEVAL — casos onde se sai bem:')
for query, comentario in queries_semanticas:
    results = dense_retrieve(query, top_k=3)
    print(f'\nQuery: "{query}" ({comentario})')
    for i, (idx, score, texto) in enumerate(results):
        print(f'  [{i+1}] {score:.3f}: {texto[:70]}')

## 3.2 Sparse Retrieval (BM25)

In [ ]:
def sparse_retrieve_bm25(query, top_k=5):
    q_tokens = re.sub(r'[^a-z0-9 ]', '', query.lower()).split()
    scores = bm25.get_scores(q_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(idx, scores[idx], corpus[idx]) for idx in top_indices]

# Casos onde BM25 se sai bem: keywords exatas
queries_keywords = [
    ('BM25 TF-IDF ranking', 'keyword exata — BM25 encontra diretamente'),
    ('HNSW algoritmo grafo', 'termos tecnicos exatos — BM25 excele'),
]

print('BM25 RETRIEVAL — casos onde se sai bem:')
for query, comentario in queries_keywords:
    results = sparse_retrieve_bm25(query, top_k=3)
    print(f'\nQuery: "{query}" ({comentario})')
    for i, (idx, score, texto) in enumerate(results):
        print(f'  [{i+1}] {score:.3f}: {texto[:70]}')

## 3.3 Hybrid Retrieval com RRF

In [ ]:
def hybrid_rrf(query, top_k=5, k=60, alpha=0.5):
    """
    Hybrid search com Reciprocal Rank Fusion.
    alpha: peso do dense (0=so BM25, 1=so dense)
    k: constante de suavizacao RRF (default 60)
    """
    # Dense ranks
    dense = dense_retrieve(query, top_k=len(corpus))
    dense_rank = {idx: rank + 1 for rank, (idx, _, _) in enumerate(dense)}
    
    # BM25 ranks
    sparse = sparse_retrieve_bm25(query, top_k=len(corpus))
    sparse_rank = {idx: rank + 1 for rank, (idx, _, _) in enumerate(sparse)}
    
    # RRF fusion
    all_ids = set(range(len(corpus)))
    fused = {}
    for doc_id in all_ids:
        d_rrf = 1.0 / (k + dense_rank.get(doc_id, len(corpus) + k))
        s_rrf = 1.0 / (k + sparse_rank.get(doc_id, len(corpus) + k))
        fused[doc_id] = alpha * d_rrf + (1 - alpha) * s_rrf
    
    top_ids = sorted(fused, key=fused.get, reverse=True)[:top_k]
    return [(idx, fused[idx], corpus[idx]) for idx in top_ids]

def mmr_retrieve(query, top_k=5, lambda_mult=0.5):
    """
    Maximum Marginal Relevance: balanceia relevancia com diversidade.
    lambda_mult: 1=apenas relevancia, 0=apenas diversidade
    """
    q_vec = model.encode(query, normalize_embeddings=True)
    doc_sims = embs @ q_vec  # similaridade com query
    
    selected = []
    remaining = list(range(len(corpus)))
    
    while len(selected) < top_k and remaining:
        if not selected:
            # Primeiro: mais relevante
            best = max(remaining, key=lambda i: doc_sims[i])
        else:
            # Seguintes: relevancia - redundancia
            selected_embs = embs[selected]
            scores = []
            for i in remaining:
                relevance = doc_sims[i]
                redundancy = max(embs[i] @ selected_embs.T)
                score = lambda_mult * relevance - (1 - lambda_mult) * redundancy
                scores.append((i, score))
            best = max(scores, key=lambda x: x[1])[0]
        
        selected.append(best)
        remaining.remove(best)
    
    return [(idx, doc_sims[idx], corpus[idx]) for idx in selected]

# Comparacao
query_test = 'como funciona busca semantica eficiente'

print(f'Query: "{query_test}"\n')

print('DENSE:')
for i, (idx, score, texto) in enumerate(dense_retrieve(query_test, 3)):
    print(f'  [{i+1}] {texto[:65]}')

print('\nBM25:')
for i, (idx, score, texto) in enumerate(sparse_retrieve_bm25(query_test, 3)):
    print(f'  [{i+1}] {texto[:65]}')

print('\nHYBRID (RRF):')
for i, (idx, score, texto) in enumerate(hybrid_rrf(query_test, 3)):
    print(f'  [{i+1}] {texto[:65]}')

print('\nMMR (relevancia+diversidade):')
for i, (idx, score, texto) in enumerate(mmr_retrieve(query_test, 3)):
    print(f'  [{i+1}] {texto[:65]}')

In [ ]:
# Visualizar: como o alpha afeta o hybrid
queries_viz = [
    'algoritmo ranking TF-IDF',      # keywords — BM25 deve ser melhor
    'encontrar documentos similares', # semantico — dense deve ser melhor  
]

alphas = np.arange(0, 1.1, 0.1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground truth manual para essas queries
ground_truth = {
    'algoritmo ranking TF-IDF': {4},    # BM25 doc
    'encontrar documentos similares': {6, 7},  # HNSW, cosine
}

for ax, query in zip(axes, queries_viz):
    recalls = []
    for alpha in alphas:
        results = hybrid_rrf(query, top_k=3, alpha=alpha)
        result_ids = {idx for idx, _, _ in results}
        gt = ground_truth[query]
        recall = len(result_ids & gt) / len(gt)
        recalls.append(recall)
    
    ax.plot(alphas, recalls, 'b-o', linewidth=2, markersize=6)
    ax.axvline(x=alphas[np.argmax(recalls)], color='red', linestyle='--', alpha=0.7,
               label=f'Otimo: alpha={alphas[np.argmax(recalls)]:.1f}')
    ax.set_title(f'Query: "{query}"', fontsize=10, fontweight='bold')
    ax.set_xlabel('Alpha (0=BM25, 1=Dense)')
    ax.set_ylabel('Recall@3')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.1)

plt.suptitle('Impacto do Alpha no Hybrid Search (RRF)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Resumo

| Estrategia | Alpha | Melhor para |
|-----------|-------|-------------|
| BM25 puro | 0.0 | Dominios com jargoes tecnicos, keywords exatas |
| Hybrid (balanceado) | 0.5 | **Recomendado para RAG geral** |
| Dense puro | 1.0 | Queries conversacionais, sinonimos |
| MMR | - | Quando diversidade importa (sumarizacao, Q&A de documentos longos) |

## Proximo
- [04 — Generation Prompts](04_generation_prompts.ipynb)